# Module 30 — Supervisor / worker, and what it costs

**THE ONE IDEA:** a supervisor routes work to specialists at runtime. It is the most
common production multi-agent shape — and it costs **5 to 10x** a single agent for the
same task.

Module 05 routed to *models*. This routes to *agents*, each with its own tool surface and
system prompt. The structure looks more sophisticated. **Measure it before you believe
it.**

The single-agent baseline runs first, so the multiplier is a number and not a claim.


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
import json
from pydantic import BaseModel
from typing import Literal
from _providers import get_client
from _tools import openai_schemas, run_tool

client, MODEL, _ = get_client("openai")
PRICE_IN, PRICE_OUT = 0.40, 1.60
TASK = ("For a 250000 loan on a 320000 property: what is the year-2 early "
        "repayment charge, and is the LTV within policy for a first-time buyer?")

usage = {"in": 0, "out": 0}
def call(messages, tools=None, max_tok=500, fmt=None):
    kw = {}
    if tools: kw["tools"] = openai_schemas(tools)
    if fmt:   kw["response_format"] = fmt
    r = client.chat.completions.create(model=MODEL, max_tokens=max_tok,
                                       messages=messages, **kw)
    usage["in"] += r.usage.prompt_tokens; usage["out"] += r.usage.completion_tokens
    return r
def cost(): return usage["in"] * PRICE_IN / 1e6 + usage["out"] * PRICE_OUT / 1e6

## Baseline — one agent, both tools

In [ ]:
def tool_loop(msgs, tools, max_steps=6, max_tok=500):
    """The module-08 loop, shared by the baseline and by every worker."""
    for _ in range(max_steps):
        r = call(msgs, tools=tools, max_tok=max_tok)
        m = r.choices[0].message
        if r.choices[0].finish_reason != "tool_calls":
            return m.content
        msgs.append(m)
        for tc in m.tool_calls:
            msgs.append({"role": "tool", "tool_call_id": tc.id,
                         "content": run_tool(tc.function.name,
                                             json.loads(tc.function.arguments))})
    return "(capped)"

usage = {"in": 0, "out": 0}
ans1 = tool_loop([{"role": "user", "content": TASK}], ["search_policy", "calculate"])
single, single_cost = dict(usage), cost()
print("SINGLE:", str(ans1)[:150])
print(f"  tokens in={single['in']} out={single['out']}  ${single_cost:.6f}")

## Supervisor + two specialists

Each worker gets a **narrow** goal and a **small** tool list — that is the actual design
rule, not the backstory.

In [ ]:
class Route(BaseModel):
    worker: Literal["policy_expert", "calculator", "FINISH"]
    instruction: str

SYS = {"policy_expert": "You look up bank policy. Report only what the policy says.",
       "calculator":    "You do arithmetic. Report only the number."}

def worker(role, tools, instruction):
    return tool_loop([{"role": "system", "content": SYS[role]},
                      {"role": "user", "content": instruction}],
                     tools, max_steps=4, max_tok=300)

def supervise(max_cycles=5):
    s = Route.model_json_schema(); s["additionalProperties"] = False
    findings = []
    for c in range(1, max_cycles + 1):
        # ONE constrained call: the supervisor cannot invent a worker that has no
        # dispatch entry, which is module 05's routing lesson one level up.
        r = call([{"role": "user", "content": f"TASK: {TASK}\nFINDINGS: {findings}\n"
                   "Choose the next worker, or FINISH when you have everything."}],
                 max_tok=200, fmt={"type": "json_schema",
                                   "json_schema": {"name": "r", "strict": True, "schema": s}})
        route = Route.model_validate_json(r.choices[0].message.content)
        print(f"  cycle {c}: -> {route.worker}  {route.instruction[:52]}")
        if route.worker == "FINISH":
            break
        tools = ["search_policy"] if route.worker == "policy_expert" else ["calculate"]
        findings.append({route.worker: worker(route.worker, tools, route.instruction)[:180]})
    return findings

usage = {"in": 0, "out": 0}
findings = supervise()
multi, multi_cost = dict(usage), cost()

## The bill

In [ ]:
r = lambda a, b: a / max(b, 1e-9)
print(f"{'':10} {'in':>8} {'out':>7} {'$':>11}\n" + "-" * 40)
print(f"{'single':10} {single['in']:8} {single['out']:7} {single_cost:11.6f}")
print(f"{'multi':10} {multi['in']:8} {multi['out']:7} {multi_cost:11.6f}\n" + "-" * 40)
print(f"{'ratio':10} {r(multi['in'], single['in']):7.1f}x "
      f"{r(multi['out'], single['out']):6.1f}x {r(multi_cost, single_cost):10.1f}x")

print("""
LESSON - the supervisor pattern is the production default and it is NOT free.
Where the tokens go: the supervisor makes an LLM call PER CYCLE just to decide
who acts next; every worker re-loads its own system prompt and tool schemas from
scratch, sharing nothing; and findings accumulate in the supervisor's prompt, so
its input grows each cycle - module 11's curve, one level up.

Four levers, in the order to reach for them: (1) NAMESPACE state so each worker
sees only what it needs; (2) PROMPT CACHING on the stable prefix; (3) RIGHT-SIZE
the models - supervisor strong, workers cheap (module 05); (4) PARALLELISE
independent workers (module 21's Send). Without them, 10-50x. With them, 2-5x.

So answer this BEFORE building it: what does routing buy that ONE agent with both
tools did not already do? Here, honestly, very little - the single agent answered
the same question. Reach for multi-agent when you have 10+ tools and real context
bloat, genuinely distinct expertise, or phased work with different success
criteria. 'It feels more sophisticated' is not a reason.""")

---

**Next:** `31_multi_agent_debate_and_handoff.ipynb`